# מחולל תמונות AI מאפס - DCGAN

## מה זה עושה?

זהו מודל **DCGAN** (Deep Convolutional GAN) שבונה תמונות של ספרות בכתב יד מאפס.
המודל לומד מ-60,000 דוגמאות אמיתיות (MNIST) ואז מייצר ספרות חדשות שלא קיימות בעולם.

## איך להשתמש?

1. לחץ על **Runtime → Change runtime type → GPU** (חינם!)
2. לחץ על **Runtime → Run all** כדי להריץ הכל
3. חכה כ-10-15 דקות לאימון
4. בסוף תראה תמונות שהמודל יצר

## שלבים בקוד:

1. התקנת ספריות וטעינת נתונים
2. בניית Generator (יוצר תמונות מרעש)
3. בניית Discriminator (מבחין בין אמיתי למזויף)
4. אימון - שתי הרשתות מתחרות
5. ייצור תמונות חדשות

## שלב 1: ייבוא ספריות

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'משתמש ב: {device}')
if device.type == 'cuda':
    print(f'שם GPU: {torch.cuda.get_device_name(0)}')

## שלב 2: הגדרות (Hyperparameters)

In [ ]:
BATCH_SIZE = 128       # כמה תמונות בכל סבב
IMAGE_SIZE = 64        # גודל תמונה (64x64 פיקסלים)
CHANNELS = 1           # 1 = שחור-לבן, 3 = צבע
Z_DIM = 100            # גודל הרעש שמייצר את התמונה
NUM_EPOCHS = 20        # כמה פעמים לעבור על כל הנתונים
LEARNING_RATE = 2e-4   # קצב למידה
BETA1 = 0.5            # פרמטר של אופטימייזר Adam

## שלב 3: טעינת מאגר הנתונים MNIST

MNIST = 60,000 תמונות של ספרות בכתב יד (0-9). יורד אוטומטית.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f'נטענו {len(dataset)} תמונות')

real_batch = next(iter(loader))
plt.figure(figsize=(8, 8))
plt.title('דוגמאות אמיתיות מהדאטה')
plt.axis('off')
plt.imshow(make_grid(real_batch[0][:64], padding=2, normalize=True).permute(1, 2, 0).cpu())
plt.show()

## שלב 4: בניית ה-Generator

הרשת שמקבלת רעש אקראי ומייצרת תמונה.

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            self._block(z_dim, features * 8, 4, 1, 0),
            self._block(features * 8, features * 4, 4, 2, 1),
            self._block(features * 4, features * 2, 4, 2, 1),
            self._block(features * 2, features, 4, 2, 1),
            nn.ConvTranspose2d(features, channels, 4, 2, 1),
            nn.Tanh(),
        )

    def _block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.net(x)

generator = Generator(Z_DIM, CHANNELS).to(device)
print(generator)

## שלב 5: בניית ה-Discriminator

הרשת ש'שופטת' אם התמונה אמיתית או מזויפת.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            self._block(features, features * 2, 4, 2, 1),
            self._block(features * 2, features * 4, 4, 2, 1),
            self._block(features * 4, features * 8, 4, 2, 1),
            nn.Conv2d(features * 8, 1, 4, 1, 0),
            nn.Sigmoid(),
        )

    def _block(self, in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1).squeeze(1)

discriminator = Discriminator(CHANNELS).to(device)
print(discriminator)

## שלב 6: אתחול משקלים ואופטימייזרים

In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)

generator.apply(init_weights)
discriminator.apply(init_weights)

criterion = nn.BCELoss()
opt_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
opt_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=device)
print('מוכן לאימון!')

## שלב 7: לולאת אימון

כאן הקסם קורה - שתי הרשתות 'נלחמות' אחת בשנייה ומשתפרות יחד.

In [ ]:
img_history = []

for epoch in range(NUM_EPOCHS):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.to(device)
        bs = real.size(0)

        noise = torch.randn(bs, Z_DIM, 1, 1, device=device)
        fake = generator(noise)

        disc_real = discriminator(real)
        loss_d_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = discriminator(fake.detach())
        loss_d_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_d = (loss_d_real + loss_d_fake) / 2

        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        output = discriminator(fake)
        loss_g = criterion(output, torch.ones_like(output))

        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] Batch {batch_idx}/{len(loader)} '
                  f'Loss D: {loss_d:.4f}, Loss G: {loss_g:.4f}')

    with torch.no_grad():
        fake = generator(fixed_noise).detach().cpu()
        img_history.append(make_grid(fake, padding=2, normalize=True))

print('האימון הסתיים!')

## שלב 8: הצגת התמונות שנוצרו

In [ ]:
plt.figure(figsize=(10, 10))
plt.title('תמונות שהמודל יצר (אחרי האימון)')
plt.axis('off')
plt.imshow(img_history[-1].permute(1, 2, 0))
plt.show()

## שלב 9: ייצור תמונות חדשות בכל פעם

In [ ]:
def generate_new_images(num_images=16):
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(num_images, Z_DIM, 1, 1, device=device)
        fake = generator(noise).cpu()
    grid = make_grid(fake, nrow=4, padding=2, normalize=True)
    plt.figure(figsize=(8, 8))
    plt.title(f'{num_images} תמונות חדשות')
    plt.axis('off')
    plt.imshow(grid.permute(1, 2, 0))
    plt.show()

generate_new_images(16)

## שלב 10: שמירת המודל

כדי שתוכל להוריד אותו ולהשתמש בו אחר כך.

In [ ]:
torch.save(generator.state_dict(), 'generator.pth')
torch.save(discriminator.state_dict(), 'discriminator.pth')
print('המודלים נשמרו! תוכל להוריד אותם מהפאנל בצד שמאל.')

## מה עוד אפשר לעשות?

- שנה את `NUM_EPOCHS` למספר גבוה יותר (50-100) לאיכות טובה יותר
- החלף את MNIST ל-`datasets.FashionMNIST` בשביל בגדים
- החלף ל-`datasets.CIFAR10` והגדר `CHANNELS = 3` בשביל תמונות צבעוניות
- העלה את `IMAGE_SIZE` ל-128 לאיכות טובה יותר (יקח יותר זמן)

## סיכום

בנית כאן AI אמיתי מאפס - בלי שום מודל מוכן! 
המודל למד לבד איך נראית כתב יד וייצר תמונות חדשות שלא היו קיימות.